<a href="https://colab.research.google.com/github/KarlaMichelleSorianoSanhez/Simulacion-I/blob/main/sistemas_de_linea_de_espera_con_dos_servidores_en_serie_y_paralelo_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#SISTEMAS MEDIANTE EVENTOS DISCRETOS

##Sistemas de línea de espera con dos servidores en serie

**Nombre**: Karla Michelle Soriano Sánchez  

**Instrucción**: Programar en Jupyter Notebook el seudocódigo propuesto por Sheldon Ross para un sistema de línea de espera con dos servidores en serie.

Como prueba utilizar:

$$
\lambda = 4
$$

y

$$
\mu = 6.
$$

Comentar y documentar el código. Analizar los resultados obtenidos mediante simulación.

**Objetivo**: Implementar el algoritmo de simulación por eventos discretos propuesto por Sheldon Ross para un sistema de línea de espera con dos servidores en serie.

A partir de la simulación se registrarán las llegadas, los tiempos de servicio y las salidas de los clientes, con el fin de analizar el comportamiento del sistema y obtener medidas de desempeño mediante simulación.

###Fundamento teórico

**Sistema de línea de espera con dos servidores en serie**

Un sistema de línea de espera con dos servidores en serie está formado por dos estaciones de servicio que atienden a los clientes de manera secuencial.

Cada cliente que llega al sistema debe ser atendido primero por el servidor 1. Una vez concluido este servicio, el cliente pasa al servidor 2. Finalmente, después de ser atendido por el segundo servidor, abandona el sistema.

El esquema general del sistema es:

$$
\text{Llegadas}
\longrightarrow
\text{Servidor 1}
\longrightarrow
\text{Servidor 2}
\longrightarrow
\text{Salida}
$$

Este tipo de sistema también es conocido como una cola en tándem o secuencial.

**Distribuciones utilizadas**

Las llegadas al sistema se modelan mediante un proceso de Poisson con tasa
$
\lambda.
$

Por esta razón, los tiempos entre llegadas son variables aleatorias exponenciales.

Si $T$ representa el tiempo entre dos llegadas consecutivas, entonces

$$
T \sim Exp(\lambda).
$$

De manera análoga, los tiempos de servicio de los servidores se modelan mediante variables aleatorias exponenciales con parámetro $\mu$.

Si $Y_1$ y $Y_2$ representan los tiempos de servicio de los servidores 1 y 2, respectivamente, entonces

$$
Y_1 \sim Exp(\mu)
$$

y

$$
Y_2 \sim Exp(\mu).
$$



**Variable de tiempo**

La simulación se desarrolla mediante una variable de tiempo
$
t,
$
la cual representa el instante actual del sistema.

Cada vez que ocurre un evento, el reloj de simulación avanza hasta el tiempo asociado a dicho evento.

**Estado del sistema**

Ross define el estado del sistema mediante el par ordenado

$$
(n_1,n_2),
$$

donde:

- $n_1$ representa el número de clientes asociados al servidor 1.
- $n_2$ representa el número de clientes asociados al servidor 2.

Estos valores incluyen tanto a los clientes que están siendo atendidos como a aquellos que esperan en la fila correspondiente.

**Variables de conteo**

Durante la simulación se utilizan las siguientes variables:

$N_A$ : Número de llegadas registradas hasta el instante $t$.

$N_D$ :  Número de clientes que han abandonado completamente el sistema hasta el instante $t$.

**Variables de salida**

Para cada cliente se almacenarán los instantes más importantes de su recorrido dentro del sistema.
Dado $n \geq 1$

$
A_1(n)
$
 : Hora de llegada del cliente $n$ al sistema.

$
A_2(n)
$
 : Hora de llegada del cliente $n$ al servidor 2.

$
D(n)
$
 : Hora de salida del cliente $n$ del sistema.

**Lista de eventos**

La simulación se basa en una lista de eventos formada por

$$
(t_A,t_1,t_2),
$$

donde:

- $t_A$ es el instante (hora)  de la próxima llegada.
- $t_1$ es el instante en que finalizará el servicio actualmente en el servidor 1.
- $t_2$ es el instante en que finalizará el servicio actualmente en el servidor 2.

En cada iteración se selecciona el menor de estos tres tiempos. Dicho valor determina cuál es el siguiente evento que ocurre en el sistema.

**Eventos posibles**

De acuerdo con el algoritmo de Ross, únicamente pueden ocurrir tres tipos de eventos.

- Evento 1: Llegada de un cliente.
 Ocurre cuando

$$
t_A=\min(t_A,t_1,t_2).
$$
En este caso un nuevo cliente entra al sistema y se actualizan las variables correspondientes.

- Evento 2: Salida del servidor 1. Ocurre cuando

$$
t_1<t_A
$$

y

$$
t_1\le t_2.
$$
El cliente termina su servicio en el servidor 1 y pasa al servidor 2.


- Evento 3: Salida del servidor 2. Ocurre cuando

$$
t_2<t_A
$$

y

$$
t_2<t_1.
$$

El cliente concluye su servicio en el servidor 2 y abandona el sistema.

**Lógica de la simulación por eventos discretos**

La simulación avanza de un evento al siguiente en lugar de actualizar el sistema en intervalos regulares de tiempo.

Para ello se mantiene una lista de eventos

$$
(t_A,t_1,t_2),
$$

que contiene los tiempos programados para la próxima llegada, la próxima salida del servidor 1 y la próxima salida del servidor 2.

En cada iteración se identifica el menor de estos tiempos y el reloj de simulación avanza directamente hasta dicho instante.

Una vez ocurrido el evento, se actualizan las variables de estado y se programa el siguiente evento correspondiente.

Este procedimiento continúa hasta alcanzar la condición de terminación establecida.

### Implementación del algoritmo

### Generación de tiempos aleatorios

De acuerdo con el modelo, las llegadas siguen un proceso de Poisson con tasa
$
\lambda,
$
por lo que los tiempos entre llegadas tienen distribución exponencial.

Asimismo, los tiempos de servicio de los servidores 1 y 2 se modelan mediante variables aleatorias exponenciales con parámetro

$
\mu.
$

Para evitar repetir código, se definirán funciones que permitan generar tanto tiempos entre llegadas como tiempos de servicio.


In [7]:
import numpy as np
import random as r
from tabulate import tabulate
import matplotlib.pyplot as plt

In [8]:
# Generar tiempo entre llegadas

def generar_llegada(lamda):

    # Generar variable exponencial
    return np.random.exponential(1 / lamda)

In [9]:
# Generar tiempo de servicio

def generar_servicio(mu):

    # Generar variable exponencial
    return np.random.exponential(1 / mu)

*Inicialización del sistema*

Antes de comenzar la simulación se deben establecer las condiciones iniciales propuestas por Ross.

Inicialmente no existen clientes dentro del sistema, por lo que

$$
N_A=N_D=0
$$

y

$$
(n_1,n_2)=(0,0).
$$

Posteriormente se genera el primer tiempo entre llegadas

$$
T_0
$$

y se construye la lista inicial de eventos

$$
t_A=T_0,
\qquad
t_1=t_2=\infty.
$$

Además, se crean las estructuras donde se almacenarán las horas de llegada y salida de cada cliente.

In [10]:
# Inicializar sistema

def inicializar_sistema(lamda):

  # Tiempo actual
  t = 0

  # Variables de conteo
  NA = 0
  ND = 0

  # Estado del sistema
  n1 = 0
  n2 = 0

  # Primer tiempo entre llegadas
  T0 = generar_llegada(lamda)

  # Lista de eventos
  tA = T0
  t1 = np.inf
  t2 = np.inf

  # Variables de salida
  A1 = []
  A2 = []
  D = []

  return t, NA, ND, n1, n2, tA, t1, t2, A1, A2, D

### Caso 1: Llegada de un cliente

El primer evento considerado por Ross corresponde a la llegada de un cliente al sistema.

Este evento ocurre cuando

$$
t_A=\min(t_A,t_1,t_2).
$$

En este caso el reloj de simulación avanza hasta el instante

$$
t=t_A.
$$

Posteriormente se actualizan las variables de conteo y de estado:

$$
N_A=N_A+1
$$

y

$$
n_1=n_1+1.
$$

Después se genera un nuevo tiempo entre llegadas para programar la siguiente llegada del sistema.

Si el servidor 1 se encontraba vacío, es decir,

$$
n_1=1,
$$

el cliente inicia inmediatamente su servicio y se programa la correspondiente salida mediante un tiempo de servicio generado aleatoriamente.

Finalmente se registra la hora de llegada del cliente al sistema mediante

$$
A_1(N_A)=t.
$$



In [11]:
# Caso 1: Llegada de un cliente

def evento_llegada(t, NA, n1, tA, t1, A1, lamda, mu):

  # Avanzar el reloj al instante de llegada
  t = tA

  # Actualizar número de llegadas
  NA += 1

  # Incrementar clientes asociados al servidor 1
  n1 += 1

  # Registrar llegada al sistema
  A1.append(t)

  # Generar siguiente llegada
  Tr = generar_llegada(lamda)

  # Actualizar instante de próxima llegada
  tA = t + Tr

  # Si el servidor 1 estaba vacío,
  # el cliente inicia servicio inmediatamente
  if n1 == 1:
    # Generar tiempo de servicio
    Y1 = generar_servicio(mu)

    # Programar salida del servidor 1
    t1 = t + Y1

  return t, NA, n1, tA, t1, A1

### Caso 2: Salida del servidor 1

El segundo evento ocurre cuando el servidor 1 concluye un servicio antes que cualquier otro evento de la lista.

Esto sucede cuando

$$
t_1<t_A
$$

y

$$
t_1\le t_2.
$$

En este instante el cliente abandona el servidor 1 y pasa al servidor 2.

Por lo tanto, el número de clientes asociados al servidor 1 disminuye en una unidad, mientras que el número de clientes asociados al servidor 2 aumenta en una unidad:

$$
n_1=n_1-1
$$

$$
n_2=n_2+1.
$$

Si todavía quedan clientes esperando o siendo atendidos en el servidor 1, se programa la siguiente salida de dicho servidor.

Asimismo, si el servidor 2 estaba libre, el cliente que acaba de llegar inicia inmediatamente su servicio.

Finalmente se registra el instante de llegada al servidor 2 mediante

$$
A_2(N_A-n_1)=t.
$$

tal como se indica en el algoritmo de Ross.

In [12]:
# Caso 2: Salida del servidor 1

def salida_servidor_1(t, NA, n1, n2, t1, t2, A2, mu):

  # Avanzar reloj
  t = t1

  # Cliente abandona servidor 1
  n1 -= 1

  # Cliente entra al servidor 2
  n2 += 1

  # Registrar llegada al servidor 2

  """ # En la notación de Ross: A2(NA - n1) = t
    En Python almacenamos los tiempos
    en una lista utilizando append().

  """
  A2.append(t)

  # Actualizar servidor 1
  if n1 == 0:
    # No hay clientes en servidor 1
    t1 = np.inf

  else:
    # Programar siguiente salida del servidor 1
    Y1 = generar_servicio(mu)

    t1 = t + Y1

  # Si el servidor 2 estaba libre
  if n2 == 1:
    # Programar salida del servidor 2
    Y2 = generar_servicio(mu)

    t2 = t + Y2

  return t, n1, n2, t1, t2, A2

*Caso 3: Salida del servidor 2*

El tercer evento ocurre cuando la finalización del servicio en el servidor 2 es el siguiente evento de la lista.

Esto sucede cuando

$$
t_2<t_A
$$

y

$$
t_2<t_1.
$$

En este caso el reloj de simulación avanza hasta

$$
t=t_2.
$$

El cliente termina su servicio y abandona completamente el sistema.

Por lo tanto, se incrementa el número de salidas registradas:

$$
N_D=N_D+1,
$$

y se reduce en una unidad el número de clientes asociados al servidor 2:

$$
n_2=n_2-1.
$$

Si aún permanecen clientes en el servidor 2, se programa la siguiente salida generando un nuevo tiempo de servicio.

Finalmente se registra el instante de salida mediante

$$
D(N_D)=t,
$$

tal como se establece en el algoritmo de Ross.

In [13]:
# Caso 3: Salida del servidor 2

def salida_servidor_2(t, ND, n2, t2, D, mu):
  # Avanzar reloj
  t = t2

  # Actualizar número de salidas
  ND += 1

  # Cliente abandona el sistema
  n2 -= 1

  # Registrar salida del sistema
  """
    En la notación de Ross:
    D(ND) = t

    En Python almacenamos los tiempos
    en una lista utilizando append().
  """
  D.append(t)

  # Si todavía existen clientes en el servidor 2
  if n2 > 0:

    # Generar nuevo tiempo de servicio
    Y2 = generar_servicio(mu)

    # Programar siguiente salida
    t2 = t + Y2

  else:
    # No hay clientes en el servidor 2
    t2 = np.inf

  return t, ND, n2, t2, D

**Función principal de simulación**

Una vez implementados los tres eventos del algoritmo, se puede construir la simulación completa.

La función principal tendrá como objetivo mantener actualizada la lista de eventos

$$
(t_A,t_1,t_2),
$$

identificando en cada iteración cuál de ellos ocurre primero.

Dependiendo del evento seleccionado, se ejecutará la función correspondiente:

- `evento_llegada()`
- `salida_servidor_1()`
- `salida_servidor_2()`

De esta manera se reproduce fielmente el diagrama de flujo propuesto por Ross.